# Lab 3 — Model names, providers, quantization and modality (EP01)

In Lab 2 we pinned a single provider, because the same Gemma from different providers gave different probabilities. In this lab we find out why.

**What you'll learn**
1. Read a model name with code, and work out the memory each precision needs
2. Open Gemma's provider list: quantization, price and context window
3. Ask Gemma the same question through several providers and compare the probabilities
4. Check the modality before sending an image — Gemma accepts it, Llama 3.1 doesn't

Provider and modality data come from OpenRouter (the textbook doesn't go into this) — the numbers change with the day you run it · the chart in part 5 is **simulated data for teaching**

> Reference: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §1.8, §1.10, §1.14 — CC BY-SA 4.0

> ✏️ **Starter notebook — you write the key parts**
>
> Most of the code is ready. Write only the **4 places marked `✏️ TODO`** (replace `___` or `...` with your code). Watch the video first, then try it yourself without opening the solution.
>
> Run every cell from top to bottom. A `NameError` or `TypeError` in a cell you have not finished yet is normal.
>
> Stuck? The solution is **`ep01-lab3-model-names.en.ipynb`** (the same code as in the video).

## 0. Setup

You need Python 3.10+ (or Colab) and `OPENROUTER_API_KEY` — how to set up the key, and why we chose this model, are in **Lab 1**

In [ ]:
%pip install -q "openai>=1.50" python-dotenv requests matplotlib

In [ ]:
# ---- from Lab 1: set up the client ----
import os, io, math, base64, requests
from collections import Counter
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
LLAMA = "meta-llama/llama-3.1-8b-instruct"
EXTRA = {"reasoning": {"enabled": False}}

## 1. Reading a model name

`<maker>/<family>-<version>-<size>-…-<it/instruct>` · `26b` = 26 billion parameters · `a4b` = MoE with ~4B active per token · `it` = instruction-tuned

In [ ]:
maker, name = MODEL.split("/")
print(maker, name.split("-"))

In [ ]:
def params_b(model_id):
    for part in model_id.split("/")[1].split("-"):
        if part.endswith("b") and part[:-1].replace(".", "").isdigit():
            return float(part[:-1])

for mid in [MODEL, LLAMA]:
    print(f"{mid}: {params_b(mid)} billion parameters")

## 2. Memory and quantization

Memory (GB) ≈ parameters (billions) × bits ÷ 8 — the model itself only, not the working space used while answering

In [ ]:
# ✏️ TODO 1: Memory (GB) = parameters (billions) × bits ÷ 8
def memory_gb(params_b, bits):
    return ___

for q, bits in [("bf16", 16), ("fp8", 8), ("fp4", 4)]:
    print(f"{q}: {memory_gb(params_b(MODEL), bits):.0f} GB")

## 3. Gemma's providers

`https://openrouter.ai/api/v1/models/<model>/endpoints` lists every provider serving this model, with its quantization, price and context window

In [ ]:
url = f"https://openrouter.ai/api/v1/models/{MODEL}/endpoints"
endpoints = requests.get(url).json()["data"]["endpoints"]
for e in sorted(endpoints, key=lambda e: float(e["pricing"]["prompt"])):
    price = float(e["pricing"]["prompt"]) * 1e6
    print(f"{e['provider_name']:<14} {e['quantization']:<8} ${price:.3f}/M  context {e['context_length']:,}")

In [ ]:
# ✏️ TODO 2: Count the providers per quantization with Counter
print(len(endpoints), "provider:", Counter(___ for e in endpoints))

## 4. Same name, different answers

Ask "I enjoy eating" as in Lab 2, pinning one provider at a time, and compare the probabilities of `pizza` and `food`

In [ ]:
def first_token_probs(provider):
    resp = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": "Complete with one word only. I enjoy eating"}],
        max_tokens=1, logprobs=True, top_logprobs=5,
        # ✏️ TODO 3: Pin the given provider alone, and forbid switching to another
        extra_body={**EXTRA, "provider": ___},
    )
    top = resp.choices[0].logprobs.content[0].top_logprobs
    return {t.token: math.exp(t.logprob) for t in top}

In [ ]:
for p in ["google-vertex/global", "nextbit/bf16", "akashml/fp4", "darkbloom"]:
    probs = first_token_probs(p)
    print(f"{p:<22} pizza {probs.get('pizza', 0):.2f} · food {probs.get('food', 0):.2f}")

In [ ]:
r = client.chat.completions.create(
    model=MODEL, messages=[{"role": "user", "content": "Hi"}], max_tokens=5,
    extra_body={**EXTRA, "provider": {"quantizations": ["bf16"]}},
)
print("served by:", r.provider)

## 5. Modality: which models accept images

OpenRouter's model list has `architecture.input_modalities` (what it accepts) and `output_modalities` (what it answers in)

In [ ]:
models = requests.get("https://openrouter.ai/api/v1/models").json()["data"]
text_only = [m["id"] for m in models if m["architecture"]["input_modalities"] == ["text"]]
print(len(models), "models ·", len(text_only), "accept text only")

In [ ]:
# ✏️ TODO 4: Return True if kind is in the model's input_modalities
def accepts(model_id, kind):
    m = next(x for x in models if x["id"] == model_id)
    return ___

for mid in [MODEL, LLAMA]:
    m = next(x for x in models if x["id"] == mid)
    print(mid, "· image:", accepts(mid, "image"), "· answers in:", m["architecture"]["output_modalities"])

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4, 2.4))
ax.bar(["2021", "2022", "2023", "2024"], [12, 30, 85, 160])
ax.set_title("Papers on LLM agents")
buf = io.BytesIO()
fig.savefig(buf, format="png")
IMAGE = "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

In [ ]:
question = [
    {"type": "text", "text": "Which year has the most papers? Answer briefly."},
    {"type": "image_url", "image_url": {"url": IMAGE}},
]
for mid in [MODEL, LLAMA]:
    try:
        r = client.chat.completions.create(model=mid, messages=[{"role": "user", "content": question}],
                                           max_tokens=40, extra_body=EXTRA)
        print(mid, "→", r.choices[0].message.content, "·", r.usage.prompt_tokens, "input tokens")
    except Exception as e:
        print(mid, "→ error:", str(e)[:90])

## Summary

| What we saw | From this lab |
|---|---|
| A model name tells the maker, size and type | `google` · `26b` · `a4b` · `it` |
| Quantization cuts memory | 52 → 26 → 13 GB |
| Same name, but providers differ in price, precision and context | 15 providers · prices vary almost 4× · context 131K or 262K |
| Different providers, different answers | Google: pizza ≈ 0.76 · Darkbloom: food ≈ 0.8 |
| Check the modality before sending an image | Gemma answers 2024 (~280 tokens) · Llama error 404 |

**Exercise:** find a model on OpenRouter that accepts `audio` (use `accepts`), then read its name with `split` — does the name tell its size?

▶ **Next: the EP01 recap** — a look back at everything we saw in the three labs